# 🧠 AI Product Feedback Intelligence

A practical notebook to analyze customer reviews, estimate sentiment, discover themes, detect feature requests and pain points, rank opportunities, and export reports. The baseline needs no paid API key. Sentiment and clustering are approximate and require human validation.

In [ ]:
# If needed, uncomment:
# %pip install pandas numpy scikit-learn matplotlib

In [ ]:
from pathlib import Path
import re, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
OUT = Path("product_feedback_output"); OUT.mkdir(exist_ok=True)


## 1. Load sample data or your CSV
For your own data, set `USE_CSV = True` and point `CSV_PATH` to a CSV containing `feedback_text`.

In [ ]:
rows = [
("F01","Mobile",2,"App crashes when I upload a photo","App Store"),
("F02","Mobile",1,"Login is slow and sometimes never finishes","App Store"),
("F03","Mobile",2,"Please add dark mode for night use","Survey"),
("F04","Dashboard",5,"Dashboard is clean and easy to understand","Survey"),
("F05","Checkout",1,"Payment failed twice and I had to start again","Support"),
("F06","Checkout",2,"Checkout is slow and confirmation is confusing","App Store"),
("F07","Search",4,"Search is much better after the update","Survey"),
("F08","Notifications",2,"Too many notifications; let me choose alerts","Support"),
("F09","Mobile",1,"App freezes when I open order history","App Store"),
("F10","Checkout",5,"Payment was quick and checkout was simple","Survey"),
("F11","Search",3,"Search should support price filters","Survey"),
("F12","Dashboard",3,"Please add CSV export and customizable widgets","Survey"),
("F13","Checkout",1,"I was charged but the order did not appear","Support"),
("F14","Search",2,"Search is slow and filters reset unexpectedly","App Store"),
("F15","Mobile",5,"App is intuitive and navigation is fast","Survey"),
("F16","Notifications",4,"Settings are useful but need more alert options","App Store"),
("F17","Dashboard",2,"Dashboard loads slowly with large accounts","Support"),
("F18","Checkout",4,"Saved payment methods make checkout easier","Survey"),
("F19","Mobile",2,"Photo upload is unreliable on slow connection","Support"),
("F20","Search",4,"Results are relevant but sorting is missing","Survey"),
("F21","Notifications",1,"Alerts arrive late and I cannot control frequency","Support"),
("F22","Dashboard",5,"Dashboard gives me information at a glance","Survey"),
("F23","Mobile",3,"Please make onboarding easier for new users","Survey"),
("F24","Checkout",2,"Cart was cleared after the payment page timed out","App Store")]
demo = pd.DataFrame(rows, columns=["feedback_id","product","rating","feedback_text","channel"])
USE_CSV = False
CSV_PATH = Path("customer_feedback.csv")
df = pd.read_csv(CSV_PATH) if USE_CSV else demo.copy()
if "feedback_text" not in df.columns: raise ValueError("CSV must contain feedback_text")
for col, default in [("feedback_id",None),("product","Unknown"),("rating",np.nan),("channel","Unknown")]:
    if col not in df: df[col] = [f"F{i+1}" for i in range(len(df))] if col=="feedback_id" else default
df["feedback_text"] = df.feedback_text.fillna("").astype(str)
df = df[df.feedback_text.str.strip().ne("")].copy().reset_index(drop=True)
df["rating"] = pd.to_numeric(df.rating, errors="coerce")
df.head()

## 2. Clean text and estimate sentiment
This transparent starter lexicon is not production-grade; it may miss sarcasm, negation, and context.

In [ ]:
def clean_text(s):
    s = re.sub(r"https?://\S+|www\.\S+", " ", str(s).lower())
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s']", " ", s)).strip()
POS = set("good great excellent easy fast helpful love like clean simple intuitive useful quick better smooth reliable".split())
NEG = set("bad slow crash crashes failed fail confusing error freeze freezes unreliable late missing broken problem reset timeout charged".split())
def sentiment(s):
    tok = re.findall(r"[a-z']+", s.lower()); p=sum(t in POS for t in tok); n=sum(t in NEG for t in tok)
    score=(p-n)/max(1,p+n)
    return ("positive" if score>.15 else "negative" if score<-.15 else "neutral", round(score,3))
df["clean_text"] = df.feedback_text.map(clean_text)
sc = df.clean_text.map(sentiment)
df["sentiment"] = [x[0] for x in sc]; df["sentiment_score"] = [x[1] for x in sc]
display(df[["feedback_id","rating","sentiment","sentiment_score"]].head(10))

## 3. Explore sentiment

In [ ]:
counts = df.sentiment.value_counts().reindex(["positive","neutral","negative"], fill_value=0)
display(counts.to_frame("feedback_count"))
counts.plot(kind="bar", rot=0, title="Baseline sentiment distribution")
plt.xlabel("Sentiment"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

## 4. Discover themes with TF-IDF and K-Means
TF-IDF creates weighted text features; K-Means groups similar feedback. Inspect example comments before naming a cluster. See the [scikit-learn text clustering guide](https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html).

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=2000)
X = vectorizer.fit_transform(df.clean_text)
k = min(4, max(1, len(df)//5))
df["theme_cluster"] = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X) if len(df)>=2 and k>=2 else 0
terms = np.array(vectorizer.get_feature_names_out())
def keywords(cid, n=8):
    idx=np.where(df.theme_cluster.to_numpy()==cid)[0]
    mean=np.asarray(X[idx].mean(axis=0)).ravel()
    return terms[mean.argsort()[::-1][:n]].tolist()
themes=[]
for cid in sorted(df.theme_cluster.unique()):
    part=df[df.theme_cluster==cid]
    themes.append({"theme_cluster":int(cid),"feedback_count":len(part),"top_keywords":", ".join(keywords(int(cid))),
                   "negative_share":round(float((part.sentiment=="negative").mean()),3),
                   "average_rating":round(float(part.rating.mean()),2) if part.rating.notna().any() else np.nan})
themes=pd.DataFrame(themes).sort_values(["negative_share","feedback_count"],ascending=False)
themes

## 5. Detect feature requests and pain points

In [ ]:
FEATURE=[r"\bplease add\b",r"\bshould support\b",r"\bwould like\b",r"\blet me\b",r"\bmissing\b",r"\bmore options\b"]
PAIN=[r"\bcrash(?:es|ed)?\b",r"\bslow\b",r"\bfailed\b",r"\bfreeze(?:s|d)?\b",r"\bconfusing\b",r"\berror\b",r"\breset\b",r"\btimeout\b",r"\bcharged\b",r"\bunreliable\b",r"\btoo many\b"]
def has(text, pats): return any(re.search(p,text,re.I) for p in pats)
df["feature_request_flag"]=df.feedback_text.map(lambda x:has(x,FEATURE))
df["pain_point_flag"]=df.feedback_text.map(lambda x:has(x,PAIN))
print("Potential feature requests:",int(df.feature_request_flag.sum()))
print("Potential pain points:",int(df.pain_point_flag.sum()))
display(df.loc[df.feature_request_flag,["feedback_id","product","feedback_text"]])
display(df.loc[df.pain_point_flag,["feedback_id","product","feedback_text"]])

## 6. Prioritize opportunities
Score combines theme volume, negative sentiment share, and low-rating share. It is a triage heuristic, not a validated business KPI.

In [ ]:
op=[]
for cid in sorted(df.theme_cluster.unique()):
    part=df[df.theme_cluster==cid]; volume=len(part)
    neg=float((part.sentiment=="negative").mean())
    low=float((part.rating<=2).mean()) if part.rating.notna().any() else 0.0
    vol=min(1.0,volume/max(1,len(df)*.35))
    score=100*(.35*vol+.40*neg+.25*low)
    op.append({"theme_cluster":int(cid),"suggested_theme":" / ".join(keywords(int(cid),6)[:3]),
               "feedback_count":volume,"negative_share":round(neg,3),"low_rating_share":round(low,3),
               "priority_score_0_100":round(score,1),"examples":" | ".join(part.feedback_text.head(2))})
opportunities=pd.DataFrame(op).sort_values("priority_score_0_100",ascending=False).reset_index(drop=True)
opportunities

## 7. Optional LLM narrative prompt
Pass this evidence-grounded prompt to your chosen provider. Keep API keys in environment variables and ask the model not to invent counts.

In [ ]:
def build_llm_prompt(opportunities, feedback):
    evidence={"opportunities":opportunities.to_dict("records"),
              "feedback_examples":feedback[["feedback_id","product","feedback_text","rating","sentiment"]].head(20).to_dict("records")}
    return ("You are a product analyst. Use only this evidence. Summarize top pain points, feature requests, "
            "experiments, and caveats. Do not invent counts; separate evidence from hypotheses and cite IDs.\n"+
            json.dumps(evidence,ensure_ascii=False,indent=2,default=str))
llm_prompt=build_llm_prompt(opportunities,df)
print(llm_prompt[:2000])
# Send llm_prompt to your chosen LLM provider using its official SDK.

## 8. Export CSV, JSON, and Markdown reports

In [ ]:
def export_reports(df, themes, opportunities, out=OUT):
    out.mkdir(exist_ok=True,parents=True)
    paths={"feedback_csv":out/"feedback_with_insights.csv","themes_csv":out/"theme_summary.csv",
           "opportunities_csv":out/"product_opportunities.csv","report_json":out/"product_feedback_report.json",
           "report_md":out/"product_feedback_report.md"}
    df.to_csv(paths["feedback_csv"],index=False); themes.to_csv(paths["themes_csv"],index=False)
    opportunities.to_csv(paths["opportunities_csv"],index=False)
    payload={"feedback_count":len(df),"sentiment_distribution":df.sentiment.value_counts().to_dict(),
             "feature_request_count":int(df.feature_request_flag.sum()),"pain_point_count":int(df.pain_point_flag.sum()),
             "opportunities":opportunities.to_dict("records"),
             "limitations":["Lexicon sentiment is approximate.","Clusters need human interpretation.","Priority score is a heuristic."]}
    paths["report_json"].write_text(json.dumps(payload,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    lines=["# AI Product Feedback Intelligence Report","",f"- Feedback records: **{len(df)}**",
           f"- Feature requests flagged: **{int(df.feature_request_flag.sum())}**",
           f"- Pain points flagged: **{int(df.pain_point_flag.sum())}**","","## Sentiment distribution",""]
    lines += [f"- {k.title()}: {int(v)}" for k,v in df.sentiment.value_counts().items()]
    lines += ["","## Prioritized opportunities",""]
    for _,r in opportunities.iterrows():
        lines += [f"### {r.suggested_theme} — score {r.priority_score_0_100}/100",
                  f"- Volume: {r.feedback_count}; negative share: {r.negative_share}; low-rating share: {r.low_rating_share}",
                  f"- Examples: {r.examples}",""]
    lines += ["## Caveats","","- Sentiment may miss context and negation.","- Clusters are not verified themes.","- Review evidence before decisions."]
    paths["report_md"].write_text("\n".join(lines)+"\n",encoding="utf-8")
    return paths
exported=export_reports(df,themes,opportunities)
assert all(p.exists() for p in exported.values())
assert opportunities.priority_score_0_100.between(0,100).all()
print("Smoke tests passed.")
for key,path in exported.items(): print(key, "=>", path)

## 9. Production roadmap and responsible use

- Replace the lexicon with a validated transformer or LLM classifier.
- Add aspect-based sentiment, multilingual support, deduplication, and trend analysis.
- Build a Streamlit dashboard with product/channel/date filters.
- Add PII detection and redaction.
- Evaluate sentiment with precision, recall, and F1; review theme quality with labeled examples.
- Minimize personal data collection, redact sensitive identifiers, follow retention policies, and keep human review in the loop.
